## Simulate GPT OSS 120B as participant in code switching

### Import and process the items

In [46]:
import pandas as pd
import os, openpyxl, asyncio, anthropic
from pathlib import Path
from openai import OpenAI
from typing import Optional
import concurrent.futures


BASE_DIR = Path.cwd().parent
INPUT  = BASE_DIR / "Human_data" / "coding_clean_final.xlsx"
OUTPUT = BASE_DIR / "LLM_data" 

df_human = pd.read_excel(INPUT)

def build_participant_items(df: pd.DataFrame) -> dict:
    participant_items = {}
    for pid, group in df.groupby("ID", sort=False):
        participant_items[pid] = group[["Item", "Itemtype"]] \
                                      .to_dict(orient="records")
    return participant_items

participant_items = build_participant_items(df_human)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/xml/etree/ElementTree.py:1301: RuntimeWarning: coroutine 'run_participant' was never awaited
  self._parser.feed(data)


### Request LLM

In [47]:
# initial setup
MODEL_NAME = "Openai GPT OSS 120B"  
client = OpenAI(
    base_url="https://chat.kiconnect.nrw/api/v1",
    api_key="698db699682e09b85c8e8611:grQvUGV8jgiqLERLjgcL11ChbXzbrh1/U6SoBQX1Z60=",
    timeout=60,
    max_retries=3
)

# system prompt
SYSTEM_PROMPT = """You are a Spanish-English bilingual. You are participating in a linguistic task.

In this study, you will read sentences in English, Spanish or a mixture of Spanish and English. This study is completing sentences.

You will see sentences that are missing the last word. You are asked to complete each sentence in a plausible way with the first word that comes to mind. You are asked to complete only one word.

There are three sentence types. The following is an example of an English sentence. Here you would need to complete an English word.

Martha wants to buy groceries and goes to the ...

A possible completion could be the following:
Martha wants to buy groceries and goes to the ...
supermarket.

Una oración también puede estar completamente en español. A continuación, se muestra un ejemplo. Aquí, se escribe una palabra en español para completar la oración:
Marta quiere hacer compras y va al ...
supermercado.

Una oración también puede empezar in Spanish and change to English, as in the following example. Here you would complete an English word:
Marta quiere hacer compras and goes to the ...
supermarket.

Please write down what first comes to mind. Please complete the sentence with only one word. Please complete a Spanish sentence with a Spanish word, and the English and mixed sentences with an English word. Your completion needs to be plausible and grammatically correct, but other than that, you can write anything.

Important: only return the word."""

In [48]:
# define functions to run by participant
def run_single_trial(participant_id: str, item: dict) -> dict:
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            max_tokens=10,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": item["Item"]}
            ]
        )
        return {
            "ID":         participant_id,
            "Item":       item["Item"],
            "Itemtype":   item["Itemtype"],
            "Completion": response.choices[0].message.content.strip(),
            "status":     "success"
        }
    except Exception as e:
        return {
            "ID":         participant_id,
            "Item":       item["Item"],
            "Itemtype":   item["Itemtype"],
            "Completion": None,
            "status":     str(e)
        }

def run_participant(participant_id: str, items: list[dict],
                    max_workers: int = 5) -> list[dict]:
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(run_single_trial, participant_id, item)
            for item in items
        ]
        return [f.result() for f in concurrent.futures.as_completed(futures)]

def run_all(participant_items: dict, max_workers: int = 5) -> pd.DataFrame:
    all_results = []
    participants = list(participant_items.items())
    total = len(participants)
    
    for i, (pid, items) in enumerate(participants, 1):
        print(f"Running participant {i}/{total}: {pid}")
        results = run_participant(pid, items, max_workers)
        all_results.extend(results)
        
        # checkpoint every 10 participants
        if i % 10 == 0:
            df_checkpoint = pd.DataFrame(all_results)
            checkpoint_path = OUTPUT / f"checkpoint_{i}of{total}.csv"
            df_checkpoint.to_csv(checkpoint_path, index=False)  
            print(f"  ✓ Checkpoint: {checkpoint_path.name}")
    
    return pd.DataFrame(all_results)  

In [50]:
# Run all participants and get the results in a DataFrame
df_llm = run_all(participant_items)

Running participant 1/120: 1750873464
Running participant 2/120: 1750874407
Running participant 3/120: 1750875716
Running participant 4/120: 1750875995
Running participant 5/120: 1750879780
Running participant 6/120: 1750887668
Running participant 7/120: 1751968832
Running participant 8/120: 1751969612
Running participant 9/120: 1751969748
Running participant 10/120: 1751970033
  ✓ Checkpoint: checkpoint_10of120.csv
Running participant 11/120: 1751971669
Running participant 12/120: 1751973483
Running participant 13/120: 1751974229
Running participant 14/120: 1751977115
Running participant 15/120: 1751978691
Running participant 16/120: 1751978981
Running participant 17/120: 1751980278
Running participant 18/120: 1751980488
Running participant 19/120: 1751981228
Running participant 20/120: 1751981332
  ✓ Checkpoint: checkpoint_20of120.csv
Running participant 21/120: 1751981512
Running participant 22/120: 1751983542
Running participant 23/120: 1751986873
Running participant 24/120: 175198

### Check and save the output data

In [51]:
# check failed trial
failed = df_llm[df_llm["status"] != "success"]
if len(failed) > 0:
    print(f"⚠️ failed trial: {len(failed)}个")
    print(failed[["ID", "Item", "status"]])
else:
    print(f"✓ all success，in total of {len(df_llm)} trials")
# delete status column and save to excel
df_llm = df_llm.drop(columns=["status"])
df_llm.to_csv(OUTPUT / f"llm_{MODEL_NAME.replace('/', '-')}.csv", index=False)

⚠️ failed trial: 246个
              ID                                               Item  \
24    1750873464    Maria shut the front window and locked the back   
40    1750873464  Jeden Morgen flechtet Susan ihrer kleinen Toch...   
62    1750874407  Als der Strom ausfiel, sahen wir nichts mehr a...   
78    1750874407  Richard hatte Angst, nahe am Rand der Klippe z...   
125   1750875716  At the end of the trip, Pam had no more clothe...   
...          ...                                                ...   
5592  1753544708  Das Eichhörnchen versteckte ein paar Nüsse in ...   
5643  1753562979  At first the boy refused, but eventually he ch...   
5707  1753566927       After a warning, the bad boy was sent to his   
5724  1753599259  Peter painted the walls of the bird exhibit li...   
5737  1753599259              Das Kind kam auf die Welt with a rare   

                                                 status  
24    Error code: 429 - {'error': {'message': 'Too m...  
40    Err

### test

In [49]:
# ── use the first participant for testing ─────────────────────────────────────
first_pid = list(participant_items.keys())[0]
first_items = participant_items[first_pid]

for item in first_items[:3]:
    print(f"  {item}")

# ── only run this participant ───────────────────────────────────────
#test single participant
test_results = run_participant(first_pid, first_items)
df_test = pd.DataFrame(test_results)
print(df_test[["Itemtype", "Completion", "status"]])

  {'Item': 'Everyone was honking at the old man who was driving too', 'Itemtype': 'English'}
  {'Item': 'Die junge Mutter aus der Nachbarschaft fütterte her baby some warm', 'Itemtype': 'code_switching'}
  {'Item': 'The man downstairs played his stereo much too', 'Itemtype': 'English'}
          Itemtype Completion  \
0          English       loud   
1          English       hour   
2          English       fast   
3   code_switching       milk   
4          English        cry   
5   code_switching        key   
6   code_switching    answers   
7   code_switching    stripes   
8          English       sink   
9   code_switching     accent   
10         English       None   
11  code_switching      shade   
12         English       nose   
13         English     family   
14         English     escape   
15         English      fault   
16  code_switching      paper   
17  code_switching     forget   
18  code_switching     higher   
19         English     finger   
20         English  